https://github.com/cmphat/AI-Agent-Lab-UTE.git

**Code 8-puzzle theo Model-Based Reflex Agent**

In [1]:
import random
import time

GOAL = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

# model: mô hình thế giới
model = {
    "actions": ["UP", "DOWN", "LEFT", "RIGHT"],
    "goal": GOAL
}

# state: trạng thái hiện tại agent đang lưu
state = None

# action: hành động trước đó
last_action = None

# visited: bộ nhớ các trạng thái đã đi qua
visited = set()


def print_state(arr):
    for row in arr:
        for value in row:
            if value == 0:
                print("_", end=" ")
            else:
                print(value, end=" ")
        print()
    print()


def state_to_string(arr):
    return str(arr)


def copy_state(arr):
    return [row[:] for row in arr]


def find_zero(arr):
    for i in range(3):
        for j in range(3):
            if arr[i][j] == 0:
                return i, j


def is_goal(arr):
    return arr == GOAL


def create_random_state():
    numbers = [0, 1, 2, 3, 4, 5, 6, 7, 8]
    random.shuffle(numbers)

    arr = []
    index = 0

    for i in range(3):
        row = []
        for j in range(3):
            row.append(numbers[index])
            index += 1
        arr.append(row)

    return arr


# percept: trạng thái agent cảm nhận được từ môi trường
def get_percept(current_state):
    return current_state


# UPDATE-STATE(state, action, percept, model)
def update_state(state, action, percept, model):
    # Trong bài 8-puzzle, percept chính là ma trận hiện tại.
    # Agent cập nhật lại trạng thái đang lưu bằng percept mới.
    state = percept
    return state


def can_move(arr, action):
    zero_row, zero_col = find_zero(arr)

    if action == "UP":
        return zero_row > 0
    elif action == "DOWN":
        return zero_row < 2
    elif action == "LEFT":
        return zero_col > 0
    elif action == "RIGHT":
        return zero_col < 2

    return False


def apply_action(arr, action):
    new_arr = copy_state(arr)
    zero_row, zero_col = find_zero(new_arr)

    if action == "UP":
        new_arr[zero_row][zero_col], new_arr[zero_row - 1][zero_col] = \
            new_arr[zero_row - 1][zero_col], new_arr[zero_row][zero_col]

    elif action == "DOWN":
        new_arr[zero_row][zero_col], new_arr[zero_row + 1][zero_col] = \
            new_arr[zero_row + 1][zero_col], new_arr[zero_row][zero_col]

    elif action == "LEFT":
        new_arr[zero_row][zero_col], new_arr[zero_row][zero_col - 1] = \
            new_arr[zero_row][zero_col - 1], new_arr[zero_row][zero_col]

    elif action == "RIGHT":
        new_arr[zero_row][zero_col], new_arr[zero_row][zero_col + 1] = \
            new_arr[zero_row][zero_col + 1], new_arr[zero_row][zero_col]

    return new_arr


# Heuristic đơn giản: đếm số ô sai vị trí
def heuristic(arr):
    wrong = 0

    for i in range(3):
        for j in range(3):
            if arr[i][j] != 0 and arr[i][j] != GOAL[i][j]:
                wrong += 1

    return wrong


# RULE-MATCH(state, rules)
def rule_match(state, model, visited):
    possible_actions = []

    for action in model["actions"]:
        if can_move(state, action):
            next_state = apply_action(state, action)

            if state_to_string(next_state) not in visited:
                score = heuristic(next_state)
                possible_actions.append((score, action, next_state))

    if len(possible_actions) == 0:
        return None

    # Chọn hành động làm trạng thái gần goal nhất
    possible_actions.sort(key=lambda x: x[0])

    best_action = possible_actions[0][1]

    rule = {
        "condition": "Chọn bước đi hợp lệ, chưa đi qua, và gần trạng thái đích hơn",
        "action": best_action
    }

    return rule

# MODEL-BASED-REFLEX-AGENT(percept) returns action
def model_based_reflex_agent(percept):
    global state, last_action, visited, model

    state = update_state(state, last_action, percept, model)

    rule = rule_match(state, model, visited)

    if rule is None:
        return "STOP"

    action = rule["action"]
    last_action = action

    return action
# MAIN PROGRAM
current_state = create_random_state()
visited.add(state_to_string(current_state))

steps = 0
start_time = time.time()

print("Trạng thái ban đầu:")
print_state(current_state)

while not is_goal(current_state):
    percept = get_percept(current_state)

    action = model_based_reflex_agent(percept)

    print("Agent chọn hành động:", action)

    if action == "STOP":
        print("Không còn bước đi hợp lệ hoặc bị kẹt.")
        break

    current_state = apply_action(current_state, action)
    visited.add(state_to_string(current_state))

    steps += 1

    print("Trạng thái sau khi đi:")
    print_state(current_state)

    print("Số bước:", steps)
    time.sleep(0.5)

    if steps >= 100:
        print("Dừng vì vượt quá 100 bước.")
        break

end_time = time.time()

if is_goal(current_state):
    print("Chúc mừng! Puzzle đã về trạng thái đích.")
else:
    print("Puzzle chưa giải xong.")

print("Thống kê:")
print("Số bước:", steps)
print("Số trạng thái đã nhớ:", len(visited))
print("Thời gian chạy:", round(end_time - start_time, 4), "giây")

Trạng thái ban đầu:
2 1 4 
_ 6 7 
8 3 5 

Agent chọn hành động: UP
Trạng thái sau khi đi:
_ 1 4 
2 6 7 
8 3 5 

Số bước: 1
Agent chọn hành động: RIGHT
Trạng thái sau khi đi:
1 _ 4 
2 6 7 
8 3 5 

Số bước: 2
Agent chọn hành động: DOWN
Trạng thái sau khi đi:
1 6 4 
2 _ 7 
8 3 5 

Số bước: 3
Agent chọn hành động: DOWN
Trạng thái sau khi đi:
1 6 4 
2 3 7 
8 _ 5 

Số bước: 4
Agent chọn hành động: LEFT
Trạng thái sau khi đi:
1 6 4 
2 3 7 
_ 8 5 

Số bước: 5
Agent chọn hành động: UP
Trạng thái sau khi đi:
1 6 4 
_ 3 7 
2 8 5 

Số bước: 6
Agent chọn hành động: RIGHT
Trạng thái sau khi đi:
1 6 4 
3 _ 7 
2 8 5 

Số bước: 7
Agent chọn hành động: UP
Trạng thái sau khi đi:
1 _ 4 
3 6 7 
2 8 5 

Số bước: 8
Agent chọn hành động: RIGHT
Trạng thái sau khi đi:
1 4 _ 
3 6 7 
2 8 5 

Số bước: 9
Agent chọn hành động: DOWN
Trạng thái sau khi đi:
1 4 7 
3 6 _ 
2 8 5 

Số bước: 10
Agent chọn hành động: LEFT
Trạng thái sau khi đi:
1 4 7 
3 _ 6 
2 8 5 

Số bước: 11
Agent chọn hành động: UP
Trạng thái sau khi đi

**Máy hút bụi — Simple Reflex Agent**

In [2]:
import random
import time

# 0 = bẩn
# 1 = sạch

rows = 4
cols = 4

room = []

for i in range(rows):
    row = []
    for j in range(cols):
        row.append(random.randint(0, 1))
    room.append(row)

x = 0
y = 0


def print_room():
    for i in range(rows):
        for j in range(cols):
            if i == x and j == y:
                print("V", end=" ")
            else:
                print(room[i][j], end=" ")
        print()
    print()


def get_percept(x, y):
    return room[x][y]


def simple_reflex_agent(percept):
    if percept == 0:
        return "SUCK"
    else:
        return "NO_OP"


def execute_action(action):
    global room, x, y

    if action == "SUCK":
        print(f"Ô ({x}, {y}) bẩn -> hút bụi")
        room[x][y] = 1
    elif action == "NO_OP":
        print(f"Ô ({x}, {y}) sạch -> không làm gì")


print("Ma trận ban đầu:")
print_room()

for i in range(rows):
    for j in range(cols):
        x = i
        y = j

        print(f"Robot đang ở ô ({x}, {y})")

        percept = get_percept(x, y)

        action = simple_reflex_agent(percept)

        print("Percept:", percept)
        print("Action:", action)

        execute_action(action)

        print_room()

        time.sleep(0.5)

print("Ma trận sau khi hút bụi:")
print_room()

Ma trận ban đầu:
V 0 1 1 
1 0 1 0 
1 1 0 1 
1 0 1 0 

Robot đang ở ô (0, 0)
Percept: 0
Action: SUCK
Ô (0, 0) bẩn -> hút bụi
V 0 1 1 
1 0 1 0 
1 1 0 1 
1 0 1 0 

Robot đang ở ô (0, 1)
Percept: 0
Action: SUCK
Ô (0, 1) bẩn -> hút bụi
1 V 1 1 
1 0 1 0 
1 1 0 1 
1 0 1 0 

Robot đang ở ô (0, 2)
Percept: 1
Action: NO_OP
Ô (0, 2) sạch -> không làm gì
1 1 V 1 
1 0 1 0 
1 1 0 1 
1 0 1 0 

Robot đang ở ô (0, 3)
Percept: 1
Action: NO_OP
Ô (0, 3) sạch -> không làm gì
1 1 1 V 
1 0 1 0 
1 1 0 1 
1 0 1 0 

Robot đang ở ô (1, 0)
Percept: 1
Action: NO_OP
Ô (1, 0) sạch -> không làm gì
1 1 1 1 
V 0 1 0 
1 1 0 1 
1 0 1 0 

Robot đang ở ô (1, 1)
Percept: 0
Action: SUCK
Ô (1, 1) bẩn -> hút bụi
1 1 1 1 
1 V 1 0 
1 1 0 1 
1 0 1 0 

Robot đang ở ô (1, 2)
Percept: 1
Action: NO_OP
Ô (1, 2) sạch -> không làm gì
1 1 1 1 
1 1 V 0 
1 1 0 1 
1 0 1 0 

Robot đang ở ô (1, 3)
Percept: 0
Action: SUCK
Ô (1, 3) bẩn -> hút bụi
1 1 1 1 
1 1 1 V 
1 1 0 1 
1 0 1 0 

Robot đang ở ô (2, 0)
Percept: 1
Action: NO_OP
Ô (2, 0) sạch -

**Máy hút bụi — Model-Based Reflex Agent**

In [4]:
import random
import time

# 0 = bẩn
# 1 = sạch

rows = 4
cols = 4

room = []

for i in range(rows):
    row = []
    for j in range(cols):
        row.append(random.randint(0, 1))
    room.append(row)

# persistent
state = {
    "x": 0,
    "y": 0,
    "known_room": [[None for j in range(cols)] for i in range(rows)]
}

model = {
    "rows": rows,
    "cols": cols,
    "actions": ["SUCK", "RIGHT", "DOWN", "NO_OP"]
}

rules = [
    "IF current cell is dirty THEN SUCK",
    "IF current cell is clean AND not at end of row THEN RIGHT",
    "IF at end of row AND not at last row THEN DOWN",
    "IF all cells checked THEN NO_OP"
]

last_action = None


def print_room():
    x = state["x"]
    y = state["y"]

    for i in range(rows):
        for j in range(cols):
            if i == x and j == y:
                print("V", end=" ")
            else:
                print(room[i][j], end=" ")
        print()
    print()


def get_percept():
    x = state["x"]
    y = state["y"]

    return room[x][y]


def update_state(state, action, percept, model):
    x = state["x"]
    y = state["y"]

    # Cập nhật trạng thái ô hiện tại vào bộ nhớ
    state["known_room"][x][y] = percept

    return state


def rule_match(state, rules):
    x = state["x"]
    y = state["y"]

    current_status = room[x][y]

    # Rule 1
    if current_status == 0:
        return {
            "condition": "Ô hiện tại bẩn",
            "action": "SUCK"
        }

    # Rule 2
    if current_status == 1 and y < cols - 1:
        return {
            "condition": "Ô hiện tại sạch và chưa đến cuối hàng",
            "action": "RIGHT"
        }

    # Rule 3
    if current_status == 1 and y == cols - 1 and x < rows - 1:
        return {
            "condition": "Đã đến cuối hàng, chuyển xuống hàng dưới",
            "action": "DOWN"
        }

    # Rule 4
    return {
        "condition": "Đã kiểm tra hết ma trận",
        "action": "NO_OP"
    }


def model_based_reflex_agent(percept):
    global state, last_action, model, rules

    state = update_state(state, last_action, percept, model)

    rule = rule_match(state, rules)

    action = rule["action"]

    last_action = action

    return action


def execute_action(action):
    global room, state

    x = state["x"]
    y = state["y"]

    if action == "SUCK":
        print(f"Ô ({x}, {y}) bẩn -> hút bụi")
        room[x][y] = 1
        state["known_room"][x][y] = 1

    elif action == "RIGHT":
        print("Di chuyển sang phải")
        state["y"] += 1

    elif action == "DOWN":
        print("Di chuyển xuống hàng dưới")
        state["x"] += 1
        state["y"] = 0

    elif action == "NO_OP":
        print("Không làm gì / kết thúc")


print("Ma trận ban đầu:")
print_room()

step = 0

while True:
    x = state["x"]
    y = state["y"]

    print(f"Robot đang ở ô ({x}, {y})")

    percept = get_percept()

    print("Percept:", percept)

    action = model_based_reflex_agent(percept)

    print("Action:", action)

    execute_action(action)

    print_room()

    step += 1

    if action == "NO_OP":
        break

    time.sleep(0.5)

print("Ma trận sau khi hút bụi:")
print_room()

print("Số bước:", step)

print("Bộ nhớ của agent:")
for row in state["known_room"]:
    print(row)

Ma trận ban đầu:
V 0 0 1 
0 1 0 0 
0 1 1 1 
1 1 0 0 

Robot đang ở ô (0, 0)
Percept: 1
Action: RIGHT
Di chuyển sang phải
1 V 0 1 
0 1 0 0 
0 1 1 1 
1 1 0 0 

Robot đang ở ô (0, 1)
Percept: 0
Action: SUCK
Ô (0, 1) bẩn -> hút bụi
1 V 0 1 
0 1 0 0 
0 1 1 1 
1 1 0 0 

Robot đang ở ô (0, 1)
Percept: 1
Action: RIGHT
Di chuyển sang phải
1 1 V 1 
0 1 0 0 
0 1 1 1 
1 1 0 0 

Robot đang ở ô (0, 2)
Percept: 0
Action: SUCK
Ô (0, 2) bẩn -> hút bụi
1 1 V 1 
0 1 0 0 
0 1 1 1 
1 1 0 0 

Robot đang ở ô (0, 2)
Percept: 1
Action: RIGHT
Di chuyển sang phải
1 1 1 V 
0 1 0 0 
0 1 1 1 
1 1 0 0 

Robot đang ở ô (0, 3)
Percept: 1
Action: DOWN
Di chuyển xuống hàng dưới
1 1 1 1 
V 1 0 0 
0 1 1 1 
1 1 0 0 

Robot đang ở ô (1, 0)
Percept: 0
Action: SUCK
Ô (1, 0) bẩn -> hút bụi
1 1 1 1 
V 1 0 0 
0 1 1 1 
1 1 0 0 

Robot đang ở ô (1, 0)
Percept: 1
Action: RIGHT
Di chuyển sang phải
1 1 1 1 
1 V 0 0 
0 1 1 1 
1 1 0 0 

Robot đang ở ô (1, 1)
Percept: 1
Action: RIGHT
Di chuyển sang phải
1 1 1 1 
1 1 V 0 
0 1 1 1 
1 1 0 